# Hardware Parts Semantic Segmentation

Label every pixel of an image with the type of hardware part it belongs to: hex nut, washer, bolt, ball bearing, spring, o-ring, or background.

**The data**
- Training images come with masks. A mask is an image where each pixel value is a class number (0 to 6).
- Test images have no masks. The task is to predict them.

**The metric**

The score is the mean **Dice score** over the 6 part classes. Dice measures how much the predicted region overlaps the true region (1.0 is a perfect overlap). Predicted regions with fewer than 30 pixels are treated as empty, so tiny false detections do not hurt the score.

**Approach**
- Model: an **Attention Residual U-Net**. It is a U-Net (an encoder that shrinks the image, then a decoder that grows it back to full size) with residual blocks and attention gates.
- Loss: Cross-Entropy plus Dice loss, which copes better with small parts on a large background.
- Training: mixed precision, gradient clipping and a cosine-annealing learning-rate schedule.
- Prediction: Test-Time Augmentation (TTA). Each test image is predicted three times (normal, flipped left-right, flipped up-down) and the results are averaged.
- Output: masks converted to run-length encoding (RLE) in `submission.csv`.

**Notebook outline**
1. Imports
2. Settings and random seeds
3. Dataset and augmentations
4. Model
5. Loss and metric
6. RLE encoding and test-time augmentation
7. Train/validation split
8. Model, loss, optimiser and scheduler
9. Training loop
10. Prediction and submission

## 1. Imports

In [ ]:
import os
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

## 2. Settings and random seeds

All paths and hyperparameters are defined here. `CLASS_MAP` links each part class number to the class name used in the submission file.

`set_random_seeds` fixes every random number generator so that repeated runs give similar results.

In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 16
NUMBER_OF_EPOCHS = 100
LEARNING_RATE = 1e-3
IMAGE_SIZE = 384
NUMBER_OF_CLASSES = 7             # class 0 = background, classes 1-6 = hardware parts
MIN_PIXELS_PER_PART = 30          # smaller predicted regions are ignored as noise

DATASET_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset"
TRAIN_IMAGE_DIR = f"{DATASET_DIR}/train/images"
TRAIN_MASK_DIR = f"{DATASET_DIR}/train/masks"
TEST_IMAGE_DIR = f"{DATASET_DIR}/test/images"
SAMPLE_SUBMISSION_PATH = f"{DATASET_DIR}/sample_submission.csv"
OUTPUT_SUBMISSION_PATH = "submission.csv"
BEST_MODEL_PATH = "best_unet.pth"

CLASS_MAP = {
    1: "hex_nut",
    2: "washer",
    3: "bolt",
    4: "ball_bearing",
    5: "spring",
    6: "o_ring",
}

In [ ]:
def set_random_seeds(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


set_random_seeds(RANDOM_SEED)

# Use the GPU if one is available, otherwise the CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

use_mixed_precision = torch.cuda.is_available()

## 3. Dataset and augmentations

`HardwareDataset` loads one image (and its mask, when masks are available) at a time.

- **Training augmentations**: random flips, 90-degree rotations, small shifts/zooms/rotations, and brightness/contrast/saturation changes. These make the model more robust because the parts can appear in any position or orientation.
- **Validation and test**: the image is only resized and normalised. Normalisation uses the standard ImageNet mean and standard deviation.

In [ ]:
class HardwareDataset(Dataset):
    def __init__(self, image_dir, mask_dir=None, image_names=None, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir              # None for the test set
        self.transform = transform

        if image_names is None:
            self.image_names = sorted(os.listdir(image_dir))
        else:
            self.image_names = image_names

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        image_path = os.path.join(self.image_dir, image_name)
        image = np.array(Image.open(image_path).convert("RGB"))

        # Test set: there is no mask, so return only the image and its name.
        if self.mask_dir is None:
            if self.transform is not None:
                image = self.transform(image=image)["image"]
            return image, image_name

        # Training / validation: load the mask. Each pixel value is already the class number (0 to 6).
        mask_path = os.path.join(self.mask_dir, image_name)
        mask = np.array(Image.open(mask_path), dtype=np.int64)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        return image, mask.long(), image_name

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.0625, 0.0625), rotate=(-45, 45), p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

validation_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

## 4. Model: Attention Residual U-Net

The network is built from three parts:

- **`ResidualBlock`**: two 3x3 convolutions whose output is added to the block's input (a "shortcut"). This helps gradients flow and makes deep networks easier to train. When the input and output have a different number of channels, a 1x1 convolution adjusts the shortcut.
- **`AttentionGate`**: sits on each skip connection. It looks at the decoder features (`gate_features`) and decides which locations in the encoder features (`skip_features`) matter, scaling unimportant locations (such as background) towards zero.
- **`AttentionResUNet`**: the full network.
  - The **encoder** has 5 levels. Each level halves the image size and doubles the number of channels (32, 64, 128, 256, 512).
  - The **decoder** has 4 levels. Each level doubles the image size, joins the attention-filtered encoder features of the same size, and applies a residual block.
  - A final 1x1 convolution produces one score per class for every pixel.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        # The shortcut only needs a convolution when the number of channels changes.
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, block_input):
        main_output = self.main_path(block_input)
        shortcut_output = self.shortcut(block_input)
        return functional.relu(main_output + shortcut_output, inplace=True)

In [ ]:
class AttentionGate(nn.Module):
    """Decides which parts of the skip-connection features are worth keeping."""

    def __init__(self, gate_channels, skip_channels, intermediate_channels):
        super().__init__()
        self.transform_gate_features = nn.Sequential(
            nn.Conv2d(gate_channels, intermediate_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(intermediate_channels),
        )
        self.transform_skip_features = nn.Sequential(
            nn.Conv2d(skip_channels, intermediate_channels, kernel_size=1, bias=True),
            nn.BatchNorm2d(intermediate_channels),
        )
        # Produces one importance value between 0 and 1 for every pixel location
        self.importance_map = nn.Sequential(
            nn.Conv2d(intermediate_channels, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gate_features, skip_features):
        transformed_gate = self.transform_gate_features(gate_features)
        transformed_skip = self.transform_skip_features(skip_features)

        combined = self.relu(transformed_gate + transformed_skip)
        importance = self.importance_map(combined)
        return skip_features * importance

In [ ]:
class AttentionResUNet(nn.Module):
    def __init__(self, in_channels=3, number_of_classes=7, base_channels=32):
        super().__init__()
        channels = base_channels

        # ----- Encoder: image gets smaller, channels grow -----
        self.encoder_level_1 = ResidualBlock(in_channels, channels)
        self.encoder_level_2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(channels, channels * 2))
        self.encoder_level_3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(channels * 2, channels * 4))
        self.encoder_level_4 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(channels * 4, channels * 8))
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(channels * 8, channels * 16))

        # ----- Decoder: image gets larger again, channels shrink -----
        self.upsample_4 = nn.ConvTranspose2d(channels * 16, channels * 8, kernel_size=2, stride=2)
        self.attention_4 = AttentionGate(channels * 8, channels * 8, channels * 4)
        self.decoder_level_4 = ResidualBlock(channels * 16, channels * 8)

        self.upsample_3 = nn.ConvTranspose2d(channels * 8, channels * 4, kernel_size=2, stride=2)
        self.attention_3 = AttentionGate(channels * 4, channels * 4, channels * 2)
        self.decoder_level_3 = ResidualBlock(channels * 8, channels * 4)

        self.upsample_2 = nn.ConvTranspose2d(channels * 4, channels * 2, kernel_size=2, stride=2)
        self.attention_2 = AttentionGate(channels * 2, channels * 2, channels)
        self.decoder_level_2 = ResidualBlock(channels * 4, channels * 2)

        self.upsample_1 = nn.ConvTranspose2d(channels * 2, channels, kernel_size=2, stride=2)
        self.attention_1 = AttentionGate(channels, channels, channels // 2)
        self.decoder_level_1 = ResidualBlock(channels * 2, channels)

        # One score per class for every pixel
        self.output_layer = nn.Conv2d(channels, number_of_classes, kernel_size=1)

    def forward(self, image):
        # Encoder (keep each level's output for the skip connections)
        features_1 = self.encoder_level_1(image)
        features_2 = self.encoder_level_2(features_1)
        features_3 = self.encoder_level_3(features_2)
        features_4 = self.encoder_level_4(features_3)
        bottleneck_features = self.bottleneck(features_4)

        # Decoder level 4
        decoded = self.upsample_4(bottleneck_features)
        filtered_skip = self.attention_4(gate_features=decoded, skip_features=features_4)
        decoded = torch.cat([decoded, filtered_skip], dim=1)
        decoded = self.decoder_level_4(decoded)

        # Decoder level 3
        decoded = self.upsample_3(decoded)
        filtered_skip = self.attention_3(gate_features=decoded, skip_features=features_3)
        decoded = torch.cat([decoded, filtered_skip], dim=1)
        decoded = self.decoder_level_3(decoded)

        # Decoder level 2
        decoded = self.upsample_2(decoded)
        filtered_skip = self.attention_2(gate_features=decoded, skip_features=features_2)
        decoded = torch.cat([decoded, filtered_skip], dim=1)
        decoded = self.decoder_level_2(decoded)

        # Decoder level 1
        decoded = self.upsample_1(decoded)
        filtered_skip = self.attention_1(gate_features=decoded, skip_features=features_1)
        decoded = torch.cat([decoded, filtered_skip], dim=1)
        decoded = self.decoder_level_1(decoded)

        return self.output_layer(decoded)

## 5. Loss and metric

**`CombinedLoss`** mixes two losses:
- **Cross-Entropy**: a standard per-pixel classification loss.
- **Soft Dice loss**: measures overlap per class (1 minus the Dice score) and averages it over all classes. It does not depend on how many pixels each class has, so small parts matter as much as the large background.

`dice_weight` controls the mix. At 0.5, each loss contributes half.

**`calculate_competition_dice`** is the validation metric. For every image and each of the 6 part classes:
- If both the prediction and the truth are empty, the score is 1.
- If only one of them is empty, the score is 0.
- Otherwise the score is `2 x overlap / (predicted size + true size)`.

Predicted regions smaller than `MIN_PIXELS_PER_PART` are cleared first, exactly as is done for the submission. The final result is the average of all these scores.

In [ ]:
class CombinedLoss(nn.Module):
    """Weighted sum of Cross-Entropy loss and soft Dice loss."""

    def __init__(self, number_of_classes=7, dice_weight=0.5):
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss()
        self.number_of_classes = number_of_classes
        self.dice_weight = dice_weight

    def forward(self, logits, true_masks):
        cross_entropy_loss = self.cross_entropy(logits, true_masks)

        # Turn scores into probabilities, and the true mask into one 0/1 map per class
        probabilities = torch.softmax(logits, dim=1)
        true_masks_one_hot = functional.one_hot(true_masks, num_classes=self.number_of_classes)
        true_masks_one_hot = true_masks_one_hot.permute(0, 3, 1, 2).float()

        # Dice score per class, summed over the batch and all pixels
        small_number = 1e-6      # avoids dividing by zero
        overlap = torch.sum(probabilities * true_masks_one_hot, dim=(0, 2, 3))
        total_size = torch.sum(probabilities + true_masks_one_hot, dim=(0, 2, 3))
        dice_per_class = (2.0 * overlap + small_number) / (total_size + small_number)
        dice_loss = 1.0 - torch.mean(dice_per_class)

        return (1 - self.dice_weight) * cross_entropy_loss + self.dice_weight * dice_loss

In [ ]:
def calculate_competition_dice(predicted_masks, true_masks, min_pixels=MIN_PIXELS_PER_PART):
    """Mean Dice score over every image and each of the 6 hardware-part classes."""
    dice_scores = []

    for predicted_mask, true_mask in zip(predicted_masks, true_masks):
        for class_id in range(1, 7):
            predicted_region = (predicted_mask == class_id)
            true_region = (true_mask == class_id)

            # Ignore very small predicted regions (treated as noise)
            if np.sum(predicted_region) < min_pixels:
                predicted_region = np.zeros_like(predicted_region)

            predicted_size = np.sum(predicted_region)
            true_size = np.sum(true_region)

            if predicted_size == 0 and true_size == 0:
                dice_scores.append(1.0)          # correctly predicted "not present"
            elif predicted_size == 0 or true_size == 0:
                dice_scores.append(0.0)          # one is empty and the other is not
            else:
                overlap = np.sum(predicted_region & true_region)
                dice_scores.append(2.0 * overlap / (predicted_size + true_size))

    return np.mean(dice_scores)

## 6. RLE encoding and test-time augmentation

**`rle_encode`** converts a yes/no mask into the run-length encoding used by the submission file. The mask is read column by column (Fortran order), and the result is a list of `start length` pairs, where `start` is the 1-based position of the first pixel of a run of ones and `length` is how many ones follow. Masks with fewer than `MIN_PIXELS_PER_PART` pixels give an empty string.

**`predict_with_tta`** predicts each image three times (original, flipped left-right, flipped up-down). The flipped predictions are flipped back, and the three probability maps are averaged. This gives a more stable prediction than a single pass.

In [ ]:
def rle_encode(mask, min_pixels=MIN_PIXELS_PER_PART):
    """Run-length encode a yes/no mask, reading the pixels column by column."""
    if np.sum(mask) < min_pixels:
        return ""

    # Read the mask column by column and surround it with zeros
    pixels = np.asarray(mask, dtype=np.uint8).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])

    # Every position where the value changes is the start or the end of a run
    change_positions = np.where(pixels[1:] != pixels[:-1])[0] + 1

    encoded_parts = []
    for run_number in range(0, len(change_positions), 2):
        run_start = change_positions[run_number]
        run_end = change_positions[run_number + 1]
        run_length = run_end - run_start
        encoded_parts.append(str(int(run_start)))
        encoded_parts.append(str(int(run_length)))

    return " ".join(encoded_parts)

In [ ]:
def predict_with_tta(model, images):
    """Average the predicted class probabilities of the original image and two flipped copies."""
    # Original image
    original_probabilities = functional.softmax(model(images), dim=1)

    # Flipped left-right (width is dimension 3); flip the prediction back afterwards
    horizontal_images = torch.flip(images, dims=[3])
    horizontal_probabilities = functional.softmax(model(horizontal_images), dim=1)
    horizontal_probabilities = torch.flip(horizontal_probabilities, dims=[3])

    # Flipped up-down (height is dimension 2); flip the prediction back afterwards
    vertical_images = torch.flip(images, dims=[2])
    vertical_probabilities = functional.softmax(model(vertical_images), dim=1)
    vertical_probabilities = torch.flip(vertical_probabilities, dims=[2])

    return (original_probabilities + horizontal_probabilities + vertical_probabilities) / 3.0

## 7. Train/validation split

The sorted list of training images is split 80/20: the first 80% are used for training and the last 20% for validation. Validation images are never used to update the model, so the validation Dice score shows how well the model works on unseen images.

In [ ]:
all_image_names = sorted(os.listdir(TRAIN_IMAGE_DIR))

number_of_train_images = int(len(all_image_names) * 0.8)
train_image_names = all_image_names[:number_of_train_images]
validation_image_names = all_image_names[number_of_train_images:]

train_dataset = HardwareDataset(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, train_image_names, transform=train_transform)
validation_dataset = HardwareDataset(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, validation_image_names, transform=validation_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train images: {len(train_image_names)} | Val images: {len(validation_image_names)}")

## 8. Model, loss, optimiser and scheduler

- **Optimiser**: AdamW with a small weight decay.
- **Scheduler**: cosine annealing lowers the learning rate smoothly from `LEARNING_RATE` to almost zero over all epochs.
- **Gradient scaler**: needed for mixed-precision training. Mixed precision runs parts of the model in 16-bit numbers, which is faster and uses less memory on a GPU.

In [ ]:
model = AttentionResUNet(in_channels=3, number_of_classes=NUMBER_OF_CLASSES, base_channels=32).to(device)
loss_function = CombinedLoss(number_of_classes=NUMBER_OF_CLASSES, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
learning_rate_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUMBER_OF_EPOCHS, eta_min=1e-6
)
gradient_scaler = torch.amp.GradScaler("cuda", enabled=use_mixed_precision)

total_parameters = 0
for parameter in model.parameters():
    total_parameters += parameter.numel()
print(f"Model has {total_parameters:,} trainable parameters")

## 9. Training loop

Each epoch has two steps:
1. **Training pass**: for every batch, compute the loss, update the weights, and clip the gradient size to 1.0 so a single bad batch cannot cause a huge update.
2. **Validation pass**: predict the validation images and calculate the competition Dice score.

The model is saved to `best_unet.pth` whenever the validation Dice score reaches a new best.

This is the slowest cell. Lower `NUMBER_OF_EPOCHS` in the settings cell for quicker experiments.

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0.0

    for images, masks, _ in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=use_mixed_precision):
            logits = model(images)
            loss = loss_function(logits, masks)

        gradient_scaler.scale(loss).backward()
        gradient_scaler.unscale_(optimizer)     # restore real gradient sizes before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        gradient_scaler.step(optimizer)
        gradient_scaler.update()

        total_loss += loss.item() * images.size(0)

    learning_rate_scheduler.step()
    return total_loss / len(train_image_names)


def validate_one_epoch():
    model.eval()
    predicted_masks = []
    true_masks = []

    with torch.no_grad():
        for images, masks, _ in validation_loader:
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=use_mixed_precision):
                logits = model(images)
            batch_predictions = torch.argmax(logits, dim=1).cpu().numpy()

            for prediction in batch_predictions:
                predicted_masks.append(prediction)
            for mask in masks.numpy():
                true_masks.append(mask)

    return calculate_competition_dice(predicted_masks, true_masks)

In [ ]:
best_validation_dice = 0.0

for epoch in range(NUMBER_OF_EPOCHS):
    train_loss = train_one_epoch()
    validation_dice = validate_one_epoch()

    print(f"Epoch [{epoch + 1:02d}/{NUMBER_OF_EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | Val Mean Dice: {validation_dice:.4f}")

    if validation_dice > best_validation_dice:
        best_validation_dice = validation_dice
        torch.save(model.state_dict(), BEST_MODEL_PATH)

print(f"\nBest Validation Mean Dice: {best_validation_dice:.4f}")

## 10. Prediction and submission

**Step 1: predict the test images.** Load the best saved model, then predict each test image with test-time augmentation. The predicted class map of every image is stored under the image's name.

**Step 2: build the submission file.** `sample_submission.csv` has one row per image and class, with keys like `<image_name>_<class_name>`. For each row, the key is split into the image name and the class, the matching yes/no mask is cut out of the predicted class map, and the mask is RLE-encoded. Following the sample file row by row guarantees the same row order.

In [ ]:
print("Generating predictions on Test Set with TTA...")
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

test_dataset = HardwareDataset(TEST_IMAGE_DIR, transform=validation_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

predicted_class_maps = {}     # image name -> predicted class number for every pixel

with torch.no_grad():
    for images, image_names in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        with torch.amp.autocast("cuda", enabled=use_mixed_precision):
            probabilities = predict_with_tta(model, images)
        batch_predictions = torch.argmax(probabilities, dim=1).cpu().numpy()

        for prediction, image_name in zip(batch_predictions, image_names):
            predicted_class_maps[image_name] = prediction

In [ ]:
def split_submission_key(key):
    """Split a key such as 'img_001_hex_nut' into ('img_001', 1)."""
    for class_id, class_name in CLASS_MAP.items():
        suffix = "_" + class_name
        if key.endswith(suffix):
            image_name = key[:-len(suffix)]
            return image_name, class_id
    raise ValueError(f"Could not parse class from submission key: {key}")


# Follow the sample submission row by row so the order matches exactly
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
encoded_pixels_column = []

for row_number in range(len(submission)):
    key = submission.loc[row_number, "ImageId_ClassId"]
    image_name, class_id = split_submission_key(key)

    predicted_class_map = predicted_class_maps[image_name]
    part_mask = (predicted_class_map == class_id)

    encoded_pixels_column.append(rle_encode(part_mask, min_pixels=MIN_PIXELS_PER_PART))

submission["EncodedPixels"] = encoded_pixels_column
submission.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
print(f"Successfully created '{OUTPUT_SUBMISSION_PATH}' with {len(submission)} rows.")